# **Camera Detection Component: YOLOV8 + COCO8 Dataset**

In this notebook i will train a YoloV8 model with COCO8 datasets for phone and person detection.

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU found')

2.6.0+cu124
True
NVIDIA GeForce RTX 2070


In [ ]:
import torch.multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import numpy as np
import torchvision.models as models
import shutil
import os
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader, random_split, Subset
from sklearn.model_selection import train_test_split

In [ ]:
# Force device to CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    print('Warning: CUDA not available, falling back to CPU.')
else:
    print(f'Using device: {device}')

Using device: cuda


In [ ]:
# Setup
!pip install ultralytics
import os
from ultralytics import YOLO

In [ ]:
def prepare_webcam_model(model_variant='yolov8m.pt'):
    """
    1. Description:
    Loads a pretrained YOLOv8 model and exports it to TorchScript format for optimal webcam inference.

    2. Input Arguments:
        model_variant (str, optional): The YOLOv8 model version to load. Defaults to 'yolov8m.pt'.

    3. Output Arguments:
        model (ultralytics.YOLO): The loaded YOLOv8 model instance.

    4. Example:
        model = prepare_webcam_model('yolov8s.pt')

    Sources:
    - YOLOv8: Jocher, G., et al. (2023). Ultralytics YOLOv8.
      https://github.com/ultralytics/ultralytics
    - COCO dataset: Lin, T.Y., et al. (2014). Microsoft COCO: Common Objects in Context. ECCV.
      Pretrained weights cover 80 classes including person (0) and cell phone (67).

    """
    # Load pretrained COCO weights as per proposal
    model_cam = YOLO(model_variant)

    # Save/Export the model
    model_cam.export(format='torchscript')
    return model_cam

In [ ]:
# 3. Prepare Webcam Component
model_cam = prepare_webcam_model()

Ultralytics 8.4.38  Python-3.9.13 torch-2.6.0+cu124 CPU (AMD Ryzen 7 1700 Eight-Core Processor)
YOLOv8m summary (fused): 92 layers, 25,886,080 parameters, 0 gradients, 78.9 GFLOPs

PyTorch: starting from 'yolov8m.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (49.7 MB)

TorchScript: starting export with torch 2.6.0+cu124...
TorchScript: export success  5.7s, saved as 'yolov8m.torchscript' (99.3 MB)

Export complete (7.6s)
Results saved to C:\Users\angel
Predict:         yolo predict task=detect model=yolov8m.torchscript imgsz=640 
Validate:        yolo val task=detect model=yolov8m.torchscript imgsz=640 data=coco.yaml  
Visualize:       https://netron.app


In [ ]:
# # Download the weights (yolov8m.pt) for local inference
# from google.colab import files
# files.download('yolov8m.pt')

### 4. Evaluate the model with testing metrics

To evaluate the model, we can use the `model.val()` method. This will run the validation process on a specified dataset (COCO in this case) and output various performance metrics.

In [ ]:
# Evaluate the model's performance on the COCO dataset for specific classes
# Class IDs: 0 for person, 67 for cell phone in COCO dataset
metrics = model_cam.val(
    data='coco.yaml',
    classes=[0, 67]
)

# Print some key metrics
print(f"Map50-95: {metrics.box.map}")
print(f"Map50: {metrics.box.map50}")
print(f"Map75: {metrics.box.map75}")

Ultralytics 8.4.38  Python-3.9.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2070, 8192MiB)
YOLOv8m summary (fused): 92 layers, 25,886,080 parameters, 0 gradients, 78.9 GFLOPs
val: Fast image access  (ping: 1.10.4 ms, read: 82.526.1 MB/s, size: 148.5 KB)
val: Scanning C:\Users\angel\datasets\coco\labels\val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 2.1it/s 2:27
                   all       5000      11039      0.776      0.646       0.72      0.521
                person       2693      10777      0.842       0.73      0.827       0.61
            cell phone        214        262      0.709      0.561      0.613      0.432
Speed: 1.2ms preprocess, 19.4ms inference, 0.0ms loss, 1.9ms postprocess per image
Saving C:\Users\angel\runs\detect\val6\predictions.json...

Evaluating faster-coco-eval mAP using C:\Users\angel\runs\detec

In [ ]:
# Define classes based on the validation call: 0 is person, 67 is cell phone
class_names = ['Person', 'Cell Phone']

print("Validation Metrics Per Class:")
print("-" * 30)
for i, name in enumerate(class_names):
    precision = metrics.box.p[i]
    recall = metrics.box.r[i]
    print(f"{name:<12} | Precision: {precision:.4f} | Recall: {recall:.4f}")

print("-" * 30)
print(f"Overall mAP50-95: {metrics.box.map:.4f}")

Validation Metrics Per Class:
------------------------------
Person       | Precision: 0.8418 | Recall: 0.7304
Cell Phone   | Precision: 0.7095 | Recall: 0.5611
------------------------------
Overall mAP50-95: 0.5210


### Confusion Matrix

In [ ]:
import numpy as np

if hasattr(metrics, 'confusion_matrix'):
    # Map COCO indices: 0 is Person, 67 is Cell Phone
    indices = [0, 67]
    conf_matrix = metrics.confusion_matrix.matrix[indices][:, indices]

    print("Confusion Matrix (Text Format)")
    print("-" * 40)
    print(f"{'':<15} | {'Pred Person':<12} | {'Pred Phone':<12}")
    print("-" * 40)
    print(f"{'Actual Person':<15} | {int(conf_matrix[0][0]):<12} | {int(conf_matrix[0][1]):<12}")
    print(f"{'Actual Phone':<15} | {int(conf_matrix[1][0]):<12} | {int(conf_matrix[1][1]):<12}")
    print("-" * 40)
else:
    print("Standard confusion matrix data not found in metrics.")

Confusion Matrix (Text Format)
----------------------------------------
                | Pred Person  | Pred Phone  
----------------------------------------
Actual Person   | 8446         | 0           
Actual Phone    | 0            | 156         
----------------------------------------


In [ ]:
!pip install torchinfo

from torchinfo import summary

# Provide a summary of the model structure and parameter counts
summary(model_cam, input_size=(1, 3, 224, 224), device=device.type)


0: 224x224 (no detections), 87.2ms
Speed: 3.5ms preprocess, 87.2ms inference, 2.5ms postprocess per image at shape (1, 3, 224, 224)
New https://pypi.org/project/ultralytics/8.4.39 available  Update with 'pip install -U ultralytics'
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1

Layer (type:depth-idx)                             Output Shape              Param #
├─DetectionModel: 1-1                              [1, 84, 8400]             --
│    └─Sequential: 2-2                             --                        (recursive)
│    │    └─Conv: 3-1                              [1, 48, 320, 320]         (1,344)
│    │    └─Detect: 3-273                          --                        (recursive)
│    │    └─Conv: 3-3                              [1, 96, 160, 160]         (41,568)
│    │    └─Detect: 3-273                          --                        (recursive)
│    │    └─C2f: 3-5                               [1, 96, 160, 160]         (110,976)
│    │    └─Detect: 3-273                          --                        (recursive)
│    │    └─C2f: 3-152                             --                        (recursive)
│    │    └─Detect: 3-273                          --                        (recursive)
│    │    └─C2f: 3-152                     

In [1]:
!pip install nbqa autopep8
!nbqa autopep8 --in-place FocusGuard_ScreenClassifier.ipynb

No such file or directory: FocusGuard_ScreenClassifier.ipynb
